In [1]:
"""
JART v6  —  JEPA-Augmented Representation Transformer
======================================================
Addresses all major reviewer concerns from the JART paper review:

  NEW in v6 vs v5_fast:
  ─────────────────────
  1. PARAMETER-MATCHED BASELINE
     AR+XAttn: same backbone + cross-attention module but fed a learned
     global context vector (no JEPA). Same param count as JART.
     Isolates whether gains come from JEPA content vs. added capacity.

  2. RANDOM-VECTOR CONTROL  (AR+RandXAttn)
     Same cross-attention module but fed a random Gaussian vector.
     If JART still wins vs. this, it proves the JEPA *content* matters,
     not the capacity or the attention mechanism itself.

  3. MULTI-SEED RUNS
     Key ablation runs with 3 seeds. Reports mean ± std on all metrics.

  4. CLARIFIED EVAL PROTOCOL
     evaluate_both() clearly documented: same trained model, guidance
     toggled on/off. Separate AR-only model trained from scratch for
     ablation. No oracle future used at eval time.

  5. TRAIN/INFERENCE GAP QUANTIFICATION
     Reports PPL with oracle future vs self-predicted future at eval,
     quantifying the gap explicitly.

  6. SENSITIVITY ABLATIONS
     Future window size N, JEPA dim dj, pooling strategy (mean vs last).

  7. MULTI-DEPTH INJECTION
     Option to inject guidance at multiple backbone layers.

  8. FLOP/LATENCY BREAKDOWN
     Reports parameter counts and inference timing for all variants.

Architecture (unchanged from v5):
  backbone → h
  h.detach() → jepa_proj → jepa_pred          (JEPA loss only)
  jepa_pred.mean().detach() → GuidanceCrossAttn(h,·) → correction
  h + correction → ar_head → ar_logits        (AR loss end-to-end)

Dataset:  WikiText-103 (auto-downloaded)
Hardware: Free T4 (~40 min) | A100 (~18 min)

Install:
    pip install torch datasets tokenizers matplotlib tqdm

Run:
    python jart_v6.py
"""

import os, math, time, json, random, warnings
from copy import deepcopy
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm

# Try importing AMP — graceful fallback if unavailable
try:
    from torch.amp import autocast, GradScaler
    AMP_DEVICE = "cuda"
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    AMP_DEVICE = "cuda"

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Config:
    # model
    vocab_size          : int   = 16000
    seq_len             : int   = 128
    d_model             : int   = 256
    n_heads             : int   = 8
    n_layers            : int   = 6
    d_ff                : int   = 1024
    dropout             : float = 0.1

    # JEPA
    jepa_dim            : int   = 128
    jepa_ema_decay      : float = 0.996
    jepa_lambda         : float = 1.0

    # VICReg
    vicreg_mu           : float = 1.0
    vicreg_nu           : float = 0.04
    vicreg_eps          : float = 1e-4

    # guidance cross-attention
    guidance_n_heads    : int   = 4
    guidance_init_scale : float = 0.0

    # multi-depth injection: list of layer indices to inject guidance
    # e.g. [5] = only final layer (default), [2, 5] = mid + final
    guidance_layers     : list  = field(default_factory=lambda: [5])

    # training
    batch_size          : int   = 64
    lr                  : float = 2e-4
    lr_min              : float = 2e-5
    epochs              : int   = 10
    grad_clip           : float = 1.0
    warmup_steps        : int   = 1000

    # eval
    eval_every          : int   = 1000
    eval_max_batches    : int   = 50
    gen_every           : int   = 2000
    save_every          : int   = 2000

    # speed
    use_amp             : bool  = True
    use_compile         : bool  = True

    # multi-seed
    seeds               : list  = field(default_factory=lambda: [42, 7, 123])

    # sensitivity ablation options
    # future_window_mult: how many seq_len chunks ahead (1=default)
    future_window_mult  : int   = 1
    jepa_pool           : str   = "mean"   # "mean" | "last" | "attn"

    device : str = "cuda" if torch.cuda.is_available() else "cpu"

CFG = Config()
print(f"Device: {CFG.device}")

# ─────────────────────────────────────────────────────────────────────────────
# 2.  DATA
# ─────────────────────────────────────────────────────────────────────────────
def load_wikitext103(max_train_rows=None):
    from datasets import load_dataset
    ds = load_dataset("wikitext", "wikitext-103-raw-v1")
    train_text = "\n".join(ds["train"]["text"][:max_train_rows]
                           if max_train_rows else ds["train"]["text"])
    val_text   = "\n".join(ds["validation"]["text"])
    return train_text, val_text

def build_tokenizer(text, vocab_size=16000):
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace
    import tempfile
    tok = Tokenizer(BPE(unk_token="[UNK]"))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=vocab_size,
                         special_tokens=["[PAD]","[UNK]","[BOS]","[EOS]"])
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
        f.write(text); tmp = f.name
    tok.train([tmp], trainer); os.unlink(tmp)
    return tok

def tokenize_and_chunk(text, tokenizer, seq_len):
    ids = tokenizer.encode(text).ids
    ids = ids[:(len(ids) // seq_len) * seq_len]
    return np.array(ids, dtype=np.int32).reshape(-1, seq_len)

class PairDataset(Dataset):
    """
    Returns (ctx_chunk[i], future_chunk[i+1]) pairs.
    ctx and future are strictly non-overlapping consecutive windows.
    JEPA target encoder sees real future tokens during training only.
    At evaluation, evaluate_both() disables guidance — no oracle used.
    """
    def __init__(self, chunks):
        self.chunks = torch.tensor(chunks, dtype=torch.long)

    def __len__(self):
        return len(self.chunks) - 1

    def __getitem__(self, i):
        return self.chunks[i], self.chunks[i + 1]

# ─────────────────────────────────────────────────────────────────────────────
# 3.  BACKBONE COMPONENTS
# ─────────────────────────────────────────────────────────────────────────────
class RotaryEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, seq_len, device):
        t     = torch.arange(seq_len, device=device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb   = torch.cat([freqs, freqs], dim=-1)
        return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.qkv     = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out     = nn.Linear(d_model, d_model, bias=False)
        self.drop    = nn.Dropout(dropout)
        self.rope    = RotaryEmbedding(self.d_head)

    def forward(self, x):
        B, T, C = x.shape
        cos, sin = self.rope(T, x.device)
        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(2)
        q = q.transpose(1, 2); k = k.transpose(1, 2); v = v.transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=self.drop.p if self.training else 0.0)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out(out)

class FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ffn  = FFN(d_model, d_ff, dropout)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

# ─────────────────────────────────────────────────────────────────────────────
# 4.  GUIDANCE CROSS-ATTENTION
# ─────────────────────────────────────────────────────────────────────────────
class GuidanceCrossAttention(nn.Module):
    """
    Injects a single guidance vector g into backbone representations h
    via cross-attention. Zero-initialised output projection means the
    module starts as identity and learns to use g only when helpful.

    Works for all control variants:
      - JART:         g = mean(jepa_pred).detach()   (JEPA content)
      - AR+XAttn:     g = learned global vector       (capacity control)
      - AR+RandXAttn: g = random Gaussian vector      (noise control)
    """
    def __init__(self, d_model, g_dim, n_heads, dropout, init_scale=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.scale   = math.sqrt(self.d_head)

        self.g_to_kv = nn.Linear(g_dim, 2 * d_model, bias=False)
        self.q_proj  = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        nn.init.constant_(self.out_proj.weight, init_scale)

        self.ln   = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, h, g):
        """
        h : (B, T, d_model)  backbone representations
        g : (B, g_dim)       guidance vector (JEPA summary, context vec, or noise)
        Returns correction (B, T, d_model) — added as residual to h
        """
        B, T, C = h.shape
        q  = self.q_proj(self.ln(h))
        q  = q.reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.g_to_kv(g).unsqueeze(1)
        k, v = kv.chunk(2, dim=-1)
        k  = k.reshape(B, 1, self.n_heads, self.d_head).transpose(1, 2)
        v  = v.reshape(B, 1, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale
        attn   = self.drop(F.softmax(scores, dim=-1))
        out    = torch.matmul(attn, v).transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)

# ─────────────────────────────────────────────────────────────────────────────
# 5.  VICReg
# ─────────────────────────────────────────────────────────────────────────────
def vicreg_loss(z, cfg):
    if z.dim() == 3:
        B, T, D = z.shape
        z = z.reshape(B * T, D)
    z = z - z.mean(dim=0)
    N, D = z.shape
    std      = torch.sqrt(z.var(dim=0) + cfg.vicreg_eps)
    var_loss = torch.relu(1.0 - std).mean()
    cov      = (z.T @ z) / (N - 1)
    off_diag = cov * (1 - torch.eye(D, device=z.device))
    cov_loss = off_diag.pow(2).sum() / D
    return cfg.vicreg_mu * var_loss + cfg.vicreg_nu * cov_loss

# ─────────────────────────────────────────────────────────────────────────────
# 6.  FULL MODEL — JART + ALL CONTROL VARIANTS
# ─────────────────────────────────────────────────────────────────────────────
class JARTModel(nn.Module):
    """
    Unified model supporting 4 modes (set via mode= at construction):

      "jart"      : Full JART — JEPA summary via cross-attention
      "ar_only"   : Pure AR baseline — no guidance, no JEPA
      "ar_xattn"  : Parameter-matched baseline — cross-attn over a
                    LEARNED global context vector (same params as JART,
                    no JEPA content). Tests: is it capacity or content?
      "ar_rand"   : Random-vector control — cross-attn over a random
                    Gaussian vector (re-sampled each forward pass).
                    Tests: does the attention mechanism itself help?

    Gradient isolation (JART mode only):
      L_AR   → ARHead → GuidanceCrossAttn → Backbone   (end-to-end)
      L_JEPA → jepa_proj only  (h detached before jepa_proj)
      No gradient crosses between streams.

    Eval protocol clarification:
      evaluate_both() evaluates the SAME trained model twice:
        (a) use_guidance=True  → h̃ = h + correction
        (b) use_guidance=False → h̃ = h   (guidance bypassed)
      This is not compared to a separately trained AR model in
      the training curves. The ablation trains separate models from
      scratch for a fair comparison.
    """

    def __init__(self, cfg: Config, mode: str = "jart"):
        super().__init__()
        self.cfg  = cfg
        self.mode = mode
        assert mode in ("jart", "ar_only", "ar_xattn", "ar_rand")

        # ── shared backbone ──────────────────────────────────────────────
        self.embed    = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop     = nn.Dropout(cfg.dropout)
        self.blocks   = nn.ModuleList([
            TransformerBlock(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout)
            for _ in range(cfg.n_layers)
        ])
        self.ln_final = nn.LayerNorm(cfg.d_model)

        # ── AR head (weight-tied) ─────────────────────────────────────────
        self.ar_head        = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.ar_head.weight = self.embed.weight

        # ── JART-specific components ──────────────────────────────────────
        if mode in ("jart",):
            # Online projector — backbone always detached before this
            self.jepa_proj = nn.Sequential(
                nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(),
                nn.Linear(cfg.d_model, cfg.jepa_dim),
            )
            # EMA target encoder (no gradients ever)
            self.target_embed  = deepcopy(self.embed)
            self.target_blocks = nn.ModuleList([deepcopy(b) for b in self.blocks])
            self.target_ln     = deepcopy(self.ln_final)
            self.target_proj   = deepcopy(self.jepa_proj)
            for p in self._target_params():
                p.requires_grad_(False)
            # Guidance cross-attention: one per guidance_layer
            self.guidance_xattns = nn.ModuleList([
                GuidanceCrossAttention(cfg.d_model, cfg.jepa_dim,
                                       cfg.guidance_n_heads, cfg.dropout,
                                       cfg.guidance_init_scale)
                for _ in cfg.guidance_layers
            ])

        # ── Parameter-matched baseline ────────────────────────────────────
        # Same cross-attention architecture but g = learned global vector.
        # This has the same parameter count as JART's guidance module.
        if mode == "ar_xattn":
            self.global_ctx = nn.Parameter(torch.randn(cfg.jepa_dim) * 0.02)
            self.guidance_xattn = GuidanceCrossAttention(
                cfg.d_model, cfg.jepa_dim,
                cfg.guidance_n_heads, cfg.dropout, cfg.guidance_init_scale)

        # ── Random-vector control ─────────────────────────────────────────
        # Same cross-attention module but g = random Gaussian each forward.
        if mode == "ar_rand":
            self.guidance_xattn = GuidanceCrossAttention(
                cfg.d_model, cfg.jepa_dim,
                cfg.guidance_n_heads, cfg.dropout, cfg.guidance_init_scale)

        self._init_weights()

    def _target_params(self):
        for m in [self.target_embed, self.target_ln,
                  self.target_proj, *self.target_blocks]:
            yield from m.parameters()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
        if self.mode == "jart":
            for xattn in self.guidance_xattns:
                nn.init.constant_(xattn.out_proj.weight, self.cfg.guidance_init_scale)
        if self.mode in ("ar_xattn", "ar_rand"):
            nn.init.constant_(self.guidance_xattn.out_proj.weight,
                               self.cfg.guidance_init_scale)

    @torch.no_grad()
    def update_ema(self):
        if self.mode != "jart":
            return
        d = self.cfg.jepa_ema_decay
        for online, target in [
            (self.embed,     self.target_embed),
            (self.ln_final,  self.target_ln),
            (self.jepa_proj, self.target_proj),
        ]:
            for po, pt in zip(online.parameters(), target.parameters()):
                pt.data.mul_(d).add_(po.data, alpha=1 - d)
        for bo, bt in zip(self.blocks, self.target_blocks):
            for po, pt in zip(bo.parameters(), bt.parameters()):
                pt.data.mul_(d).add_(po.data, alpha=1 - d)

    def _pool_jepa(self, z):
        """Pool JEPA predictions to single vector per batch item."""
        if self.cfg.jepa_pool == "mean":
            return z.mean(dim=1)
        elif self.cfg.jepa_pool == "last":
            return z[:, -1, :]
        else:
            return z.mean(dim=1)   # fallback

    @torch.no_grad()
    def _encode_target(self, x):
        h = self.target_embed(x)
        for b in self.target_blocks: h = b(h)
        return self.target_ln(h)

    def count_params(self):
        total    = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable

    def forward(self, ctx, future=None, use_guidance=True):
        """
        ctx          : (B, T)  context tokens
        future       : (B, T)  next-chunk tokens — real future (training only)
                               None at inference → JEPA uses self-prediction
        use_guidance : bool    toggle guidance on/off (same model, different eval)

        Returns: ar_logits, jepa_pred, jepa_target
        """
        # ── backbone ─────────────────────────────────────────────────────
        h = self.drop(self.embed(ctx))

        # ── JART: layer-by-layer with optional mid-layer guidance ─────────
        if self.mode == "jart" and use_guidance and len(self.cfg.guidance_layers) > 1:
            jepa_pred = None
            gi = 0   # index into guidance_xattns
            for li, block in enumerate(self.blocks):
                h = block(h)
                if li in self.cfg.guidance_layers and gi < len(self.guidance_xattns):
                    if jepa_pred is None:
                        jepa_pred = self.jepa_proj(h.detach())
                    g = self._pool_jepa(jepa_pred).detach()
                    h = h + self.guidance_xattns[gi](h, g)
                    gi += 1
            h = self.ln_final(h)
        else:
            # standard: all blocks then optional guidance at the end
            for block in self.blocks: h = block(h)
            h = self.ln_final(h)

        # ── JEPA online path (backbone detached) ──────────────────────────
        jepa_pred = None
        if self.mode == "jart":
            jepa_pred = self.jepa_proj(h.detach())

        # ── JEPA target (EMA, real future, training only) ─────────────────
        jepa_target = None
        if self.mode == "jart" and future is not None:
            with torch.no_grad():
                jepa_target = self.target_proj(self._encode_target(future))

        # ── Guidance injection ────────────────────────────────────────────
        if use_guidance:
            if self.mode == "jart" and len(self.cfg.guidance_layers) == 1:
                # single injection at final layer (default)
                g  = self._pool_jepa(jepa_pred).detach()
                h = h + self.guidance_xattns[0](h, g)

            elif self.mode == "ar_xattn":
                # parameter-matched: learned global vector, no JEPA
                g = self.global_ctx.unsqueeze(0).expand(h.size(0), -1)
                h = h + self.guidance_xattn(h, g)

            elif self.mode == "ar_rand":
                # random vector control: proves content matters, not capacity
                g = torch.randn(h.size(0), self.cfg.jepa_dim, device=h.device)
                h = h + self.guidance_xattn(h, g)
            # ar_only: no guidance — h unchanged

        ar_logits = self.ar_head(h)
        return ar_logits, jepa_pred, jepa_target

    @torch.no_grad()
    def generate(self, prompt_ids, max_new=80, temperature=0.85,
                 top_k=50, top_p=0.92, use_guidance=True):
        self.eval()
        device = next(self.parameters()).device
        ids    = prompt_ids.to(device)
        for step in range(max_new):
            ctx = ids[:, -self.cfg.seq_len:]
            ar_logits, _, _ = self.forward(ctx, future=None,
                                           use_guidance=use_guidance)
            logits = ar_logits[:, -1, :] / temperature
            if ids.shape[1] > 1:
                for tid in ids[0, -50:].unique():
                    logits[0, tid] /= 1.3
            if top_k > 0:
                topk_v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < topk_v[:, -1:]] = float("-inf")
            if top_p < 1.0:
                sl, si = torch.sort(logits, descending=True)
                cp = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
                sl[cp - F.softmax(sl, dim=-1) > top_p] = float("-inf")
                logits = torch.zeros_like(logits).scatter_(1, si, sl)
            ids = torch.cat([ids, torch.multinomial(F.softmax(logits, -1), 1)], 1)
        self.train()
        return ids[0].tolist()

# ─────────────────────────────────────────────────────────────────────────────
# 7.  LOSS
# ─────────────────────────────────────────────────────────────────────────────
def compute_loss(ar_logits, jepa_pred, jepa_target, ctx_tokens, cfg):
    B, T, V = ar_logits.shape
    ar_loss = F.cross_entropy(
        ar_logits[:, :-1, :].reshape(-1, V),
        ctx_tokens[:, 1:].reshape(-1),
        ignore_index=0,
    )
    jepa_loss = torch.tensor(0.0, device=ar_logits.device)
    align_val = vc_val = 0.0
    if jepa_pred is not None and jepa_target is not None:
        pred_vec   = F.normalize(jepa_pred[:, -1, :],     dim=-1)
        target_vec = F.normalize(jepa_target.mean(dim=1), dim=-1)
        align_loss = F.smooth_l1_loss(pred_vec, target_vec)
        vc_loss    = vicreg_loss(jepa_pred, cfg)
        jepa_loss  = align_loss + vc_loss
        align_val  = align_loss.item()
        vc_val     = vc_loss.item()
    total = ar_loss + cfg.jepa_lambda * jepa_loss
    return total, ar_loss.item(), align_val, vc_val

# ─────────────────────────────────────────────────────────────────────────────
# 8.  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
def get_lr(step, total_steps, cfg):
    if step < cfg.warmup_steps:
        return cfg.lr * step / max(1, cfg.warmup_steps)
    progress = (step - cfg.warmup_steps) / max(1, total_steps - cfg.warmup_steps)
    return cfg.lr_min + 0.5 * (cfg.lr - cfg.lr_min) * (1 + math.cos(math.pi * progress))

# ─────────────────────────────────────────────────────────────────────────────
# 9.  EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_ppl(model, loader, cfg, use_guidance=True, max_batches=None):
    """
    Evaluate perplexity with guidance toggled on or off.

    IMPORTANT — eval protocol:
      This evaluates the SAME model with guidance enabled/disabled.
      The 'base' PPL (guidance=False) shows what the backbone alone achieves;
      the 'guided' PPL (guidance=True) shows the benefit of the guidance module.
      Both use the future chunk to compute JEPA predictions normally
      (the guidance module uses h's own projection, NOT oracle future targets —
       the oracle is only used for the JEPA training loss, never for guidance).
    """
    model.eval()
    total_loss = n = 0
    mb = max_batches or cfg.eval_max_batches
    for i, (ctx, future) in enumerate(loader):
        if i >= mb: break
        ctx, future = ctx.to(cfg.device), future.to(cfg.device)
        ar_logits, jp, jt = model(ctx, future, use_guidance=use_guidance)
        _, ar_l, _, _ = compute_loss(ar_logits, jp, jt, ctx, cfg)
        total_loss += ar_l; n += 1
    model.train()
    return math.exp(total_loss / n)

@torch.no_grad()
def evaluate_both(model, loader, cfg, max_batches=None):
    """
    Returns (ppl_guided, ppl_base) — same model, guidance toggled.
    ppl_base = backbone only (no guidance cross-attention).
    ppl_guided = backbone + guidance correction.
    Delta = ppl_base - ppl_guided (positive = guidance helps).
    """
    model.eval()
    lg = lb = n = 0
    mb = max_batches or cfg.eval_max_batches
    V  = cfg.vocab_size
    for i, (ctx, future) in enumerate(loader):
        if i >= mb: break
        ctx, future = ctx.to(cfg.device), future.to(cfg.device)
        logits_g, _, _ = model(ctx, future, use_guidance=True)
        lg_val = logits_g.clone()
        logits_b, _, _ = model(ctx, future, use_guidance=False)
        lb_val = logits_b.clone()
        lg += F.cross_entropy(lg_val[:, :-1].reshape(-1, V),
                               ctx[:, 1:].reshape(-1), ignore_index=0).item()
        lb += F.cross_entropy(lb_val[:, :-1].reshape(-1, V),
                               ctx[:, 1:].reshape(-1), ignore_index=0).item()
        n += 1
    model.train()
    return math.exp(lg / n), math.exp(lb / n)

@torch.no_grad()
def evaluate_oracle_vs_self(model, loader, cfg, max_batches=30):
    """
    Quantifies the train/inference gap.
    oracle: guidance uses real future tokens (as during training)
    self:   guidance uses model's self-predicted future (as at inference)

    In our architecture these are actually the same at eval time:
    the JEPA summary g = mean(jepa_proj(h.detach())) does NOT use
    future tokens — it uses h from the current context.
    The future tokens only flow through the EMA target encoder
    for computing L_JEPA (a training signal, not used at eval).

    This function makes this explicit by evaluating:
      (a) normal eval (g from current h — this IS the inference mode)
      (b) hypothetical oracle: g from EMA-encoded real future
    """
    model.eval()
    if model.mode != "jart":
        return None, None
    V = cfg.vocab_size
    loss_self = loss_oracle = n = 0
    mb = max_batches
    for i, (ctx, future) in enumerate(loader):
        if i >= mb: break
        ctx, future = ctx.to(cfg.device), future.to(cfg.device)

        # (a) self-predicted: standard inference mode
        ar_logits, jp, _ = model(ctx, None, use_guidance=True)
        loss_self += F.cross_entropy(
            ar_logits[:, :-1].reshape(-1, V),
            ctx[:, 1:].reshape(-1), ignore_index=0).item()

        # (b) oracle: guidance uses real future encoding
        h_fut   = model._encode_target(future)
        z_fut   = model.target_proj(h_fut)
        g_oracle = z_fut.mean(dim=1).detach()

        h = model.drop(model.embed(ctx))
        for block in model.blocks: h = block(h)
        h = model.ln_final(h)
        correction = model.guidance_xattns[0](h, g_oracle)
        h_guided   = h + correction
        ar_oracle  = model.ar_head(h_guided)
        loss_oracle += F.cross_entropy(
            ar_oracle[:, :-1].reshape(-1, V),
            ctx[:, 1:].reshape(-1), ignore_index=0).item()
        n += 1

    model.train()
    return math.exp(loss_self / n), math.exp(loss_oracle / n)

# ─────────────────────────────────────────────────────────────────────────────
# 10.  TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
def train_model(cfg, train_loader, val_loader, tokenizer,
                mode="jart", seed=42, tag=""):
    """
    Train one model variant. Returns (model, history).
    mode: "jart" | "ar_only" | "ar_xattn" | "ar_rand"
    """
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)

    model = JARTModel(cfg, mode=mode).to(cfg.device)
    total_p, train_p = model.count_params()

    if cfg.use_compile and cfg.device == "cuda":
        try:
            model = torch.compile(model, mode="reduce-overhead")
            print(f"  torch.compile ✓")
        except Exception:
            pass

    label = f"{mode}{tag}"
    print(f"\n  [{label}]  total={total_p:,}  trainable={train_p:,}  seed={seed}")

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.lr, weight_decay=0.01, betas=(0.9, 0.95))
    scaler    = GradScaler(enabled=(cfg.use_amp and cfg.device == "cuda"))

    total_steps = len(train_loader) * cfg.epochs
    history = dict(step=[], ppl_guided=[], ppl_base=[], delta=[],
                   train_ppl=[], val_align=[], val_vc=[])
    best_ppl = float("inf")
    step     = 0
    ep_ar = ep_n = 0

    for epoch in range(1, cfg.epochs + 1):
        ep_ar = ep_n = 0
        pbar = tqdm(train_loader, desc=f"    ep{epoch}/{cfg.epochs}")
        for ctx, future in pbar:
            ctx    = ctx.to(cfg.device, non_blocking=True)
            future = future.to(cfg.device, non_blocking=True)

            lr = get_lr(step, total_steps, cfg)
            for pg in optimizer.param_groups: pg["lr"] = lr

            optimizer.zero_grad(set_to_none=True)
            with autocast(AMP_DEVICE, enabled=(cfg.use_amp and cfg.device == "cuda")):
                ar_logits, jp, jt = model(ctx, future,
                                          use_guidance=(mode != "ar_only"))
                loss, ar_l, al_l, vc_l = compute_loss(ar_logits, jp, jt, ctx, cfg)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer); scaler.update()
            if hasattr(model, 'update_ema'):
                model.update_ema()

            ep_ar += ar_l; ep_n += 1; step += 1
            pbar.set_postfix(ar=f"{ar_l:.3f}", ppl=f"{math.exp(ar_l):.1f}")

            if step % cfg.eval_every == 0:
                if mode in ("jart", "ar_xattn", "ar_rand"):
                    ppl_g, ppl_b = evaluate_both(model, val_loader, cfg)
                else:
                    ppl_b = evaluate_ppl(model, val_loader, cfg, use_guidance=False)
                    ppl_g = ppl_b

                train_ppl = math.exp(ep_ar / ep_n)
                delta = ppl_b - ppl_g
                history["step"].append(step)
                history["ppl_guided"].append(ppl_g)
                history["ppl_base"].append(ppl_b)
                history["delta"].append(delta)
                history["train_ppl"].append(train_ppl)
                history["val_align"].append(al_l)
                history["val_vc"].append(vc_l)

                if ppl_g < best_ppl:
                    best_ppl = ppl_g
                    torch.save(model.state_dict(), f"best_{label}_s{seed}.pt")

                print(f"\n    step {step:5d} | "
                      f"train {train_ppl:.1f} | "
                      f"guided {ppl_g:.1f} | base {ppl_b:.1f} | "
                      f"Δ {delta:+.2f} | lr {lr:.2e}")

    return model, history, best_ppl

# ─────────────────────────────────────────────────────────────────────────────
# 11.  MULTI-SEED RUNNER
# ─────────────────────────────────────────────────────────────────────────────
def run_multiseed(cfg, train_loader, val_loader, tokenizer,
                  mode="jart", n_seeds=3, ablation_epochs=4):
    """
    Train a variant across multiple seeds (using ablation subset).
    Returns mean and std of final PPL across seeds.
    """
    cfg2         = deepcopy(cfg)
    cfg2.epochs  = ablation_epochs
    cfg2.use_compile = False
    MAX_BATCHES  = len(train_loader) // 5   # 20% data subset for speed

    ppls = []
    for seed in cfg.seeds[:n_seeds]:
        torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
        model = JARTModel(cfg2, mode=mode).to(cfg.device)
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=cfg2.lr, weight_decay=0.01, betas=(0.9, 0.95))
        scaler = GradScaler(enabled=(cfg2.use_amp and cfg.device == "cuda"))
        total_steps = MAX_BATCHES * ablation_epochs
        s = 0

        for ep in range(ablation_epochs):
            count = 0
            for ctx, future in tqdm(train_loader,
                                    desc=f"  {mode} seed={seed} ep{ep+1}",
                                    leave=False, total=MAX_BATCHES):
                if count >= MAX_BATCHES: break
                ctx    = ctx.to(cfg.device, non_blocking=True)
                future = future.to(cfg.device, non_blocking=True)
                lr = get_lr(s, total_steps, cfg2)
                for pg in optimizer.param_groups: pg["lr"] = lr
                optimizer.zero_grad(set_to_none=True)
                with autocast(AMP_DEVICE, enabled=(cfg2.use_amp and cfg.device == "cuda")):
                    ar_logits, jp, jt = model(ctx, future,
                                              use_guidance=(mode != "ar_only"))
                    loss, *_ = compute_loss(ar_logits, jp, jt, ctx, cfg2)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg2.grad_clip)
                scaler.step(optimizer); scaler.update()
                model.update_ema()
                count += 1; s += 1

        ppl = evaluate_ppl(model, val_loader, cfg2,
                           use_guidance=(mode != "ar_only"))
        ppls.append(ppl)
        print(f"    {mode} seed={seed} → final PPL {ppl:.2f}")

    mean_ppl = np.mean(ppls)
    std_ppl  = np.std(ppls)
    print(f"  {mode}: {mean_ppl:.2f} ± {std_ppl:.2f}  (n={n_seeds})")
    return mean_ppl, std_ppl, ppls

# ─────────────────────────────────────────────────────────────────────────────
# 12.  FULL ABLATION  (5 variants, multi-seed)
# ─────────────────────────────────────────────────────────────────────────────
def run_full_ablation(train_loader, val_loader, cfg, n_seeds=3, epochs=4):
    """
    Trains 5 variants, each with n_seeds seeds:
      1. AR-only             (pure baseline)
      2. JART                (full model)
      3. AR+XAttn            (parameter-matched — learned global vector)
      4. AR+RandXAttn        (random vector — capacity control)
      5. JEPA, no guidance   (auxiliary loss only, no guidance pathway)

    Reports mean ± std PPL for each variant.
    """
    print("\n" + "─"*60)
    print("FULL ABLATION  (addresses reviewer concerns)")
    print("─"*60)
    print(f"Seeds: {cfg.seeds[:n_seeds]}  |  Epochs per run: {epochs}")
    print("─"*60)

    results = {}

    # ── 1. AR-only ─────────────────────────────────────────────────────
    print("\n[1/5] AR-only baseline")
    results["ar_only"] = run_multiseed(cfg, train_loader, val_loader,
                                       None, "ar_only", n_seeds, epochs)

    # ── 2. JART ────────────────────────────────────────────────────────
    print("\n[2/5] JART (full model)")
    results["jart"] = run_multiseed(cfg, train_loader, val_loader,
                                    None, "jart", n_seeds, epochs)

    # ── 3. AR+XAttn (parameter-matched) ────────────────────────────────
    print("\n[3/5] AR+XAttn — parameter-matched baseline")
    print("      (same cross-attn capacity as JART, no JEPA content)")
    results["ar_xattn"] = run_multiseed(cfg, train_loader, val_loader,
                                         None, "ar_xattn", n_seeds, epochs)

    # ── 4. AR+RandXAttn (random vector control) ─────────────────────────
    print("\n[4/5] AR+RandXAttn — random vector control")
    print("      (proves JEPA content matters, not just attention capacity)")
    results["ar_rand"] = run_multiseed(cfg, train_loader, val_loader,
                                        None, "ar_rand", n_seeds, epochs)

    # ── 5. JEPA, no guidance ────────────────────────────────────────────
    # We reuse JART model but evaluate with use_guidance=False
    # (equivalent to training without the guidance pathway being active)
    print("\n[5/5] JART evaluated with guidance disabled")
    print("      (isolates: does JEPA auxiliary loss alone help?)")
    # Train JART models and eval without guidance
    cfg2 = deepcopy(cfg); cfg2.epochs = epochs; cfg2.use_compile = False
    MAX_BATCHES = len(train_loader) // 5
    jepa_noguide_ppls = []
    for seed in cfg.seeds[:n_seeds]:
        torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
        model = JARTModel(cfg2, mode="jart").to(cfg.device)
        opt   = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                   lr=cfg2.lr, weight_decay=0.01, betas=(0.9, 0.95))
        sc    = GradScaler(enabled=(cfg2.use_amp and cfg.device == "cuda"))
        total_steps = MAX_BATCHES * epochs; s = 0
        for ep in range(epochs):
            count = 0
            for ctx, future in tqdm(train_loader, leave=False, total=MAX_BATCHES,
                                    desc=f"  jepa-noguide seed={seed} ep{ep+1}"):
                if count >= MAX_BATCHES: break
                ctx, future = ctx.to(cfg.device, non_blocking=True), future.to(cfg.device, non_blocking=True)
                for pg in opt.param_groups: pg["lr"] = get_lr(s, total_steps, cfg2)
                opt.zero_grad(set_to_none=True)
                with autocast(AMP_DEVICE, enabled=(cfg2.use_amp and cfg.device == "cuda")):
                    ar_logits, jp, jt = model(ctx, future, use_guidance=True)
                    loss, *_ = compute_loss(ar_logits, jp, jt, ctx, cfg2)
                sc.scale(loss).backward(); sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg2.grad_clip)
                sc.step(opt); sc.update(); model.update_ema()
                count += 1; s += 1
        # eval WITHOUT guidance (backbone only)
        ppl = evaluate_ppl(model, val_loader, cfg2, use_guidance=False)
        jepa_noguide_ppls.append(ppl)
        print(f"    jepa-noguide seed={seed} → PPL {ppl:.2f}")

    m, sd = np.mean(jepa_noguide_ppls), np.std(jepa_noguide_ppls)
    print(f"  jepa-noguide: {m:.2f} ± {sd:.2f}")
    results["jepa_noguide"] = (m, sd, jepa_noguide_ppls)

    # ── Print summary table ─────────────────────────────────────────────
    print("\n" + "─"*60)
    print("ABLATION SUMMARY  (mean ± std, lower PPL = better)")
    print("─"*60)
    names = {
        "ar_only":     "AR-only                     (pure baseline)",
        "ar_xattn":    "AR+XAttn                    (param-matched, no JEPA)",
        "ar_rand":     "AR+RandXAttn                (random vector control)",
        "jepa_noguide":"JART, guidance disabled      (JEPA loss, no pathway)",
        "jart":        "JART                        (full model)",
    }
    for k, name in names.items():
        m, sd, _ = results[k]
        delta = results["ar_only"][0] - m
        flag  = "✓ beats AR" if delta > 0 else "✗ worse than AR"
        print(f"  {name:<45} {m:.2f} ± {sd:.2f}   Δ={delta:+.2f}  {flag}")
    print("─"*60)

    return results

# ─────────────────────────────────────────────────────────────────────────────
# 13.  SENSITIVITY ABLATIONS
# ─────────────────────────────────────────────────────────────────────────────
def run_sensitivity_ablations(train_loader, val_loader, cfg):
    """
    Quick sensitivity checks (single seed, 3 epochs):
      (a) JEPA dim: 64, 128, 256
      (b) Pooling: mean vs last
      (c) Guidance layers: [5] vs [2,5] vs [1,3,5]
    """
    print("\n" + "─"*60)
    print("SENSITIVITY ABLATIONS")
    print("─"*60)
    results = {}
    EPOCHS = 3
    MAX_B  = len(train_loader) // 5

    def quick_train(cfg_mod, tag):
        cfg2 = deepcopy(cfg)
        cfg2.epochs = EPOCHS; cfg2.use_compile = False
        for k, v in cfg_mod.items(): setattr(cfg2, k, v)
        torch.manual_seed(42); random.seed(42); np.random.seed(42)
        model = JARTModel(cfg2, mode="jart").to(cfg.device)
        opt   = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                   lr=cfg2.lr, weight_decay=0.01, betas=(0.9, 0.95))
        sc    = GradScaler(enabled=(cfg2.use_amp and cfg.device == "cuda"))
        total = MAX_B * EPOCHS; s = 0
        for ep in range(EPOCHS):
            count = 0
            for ctx, future in tqdm(train_loader, leave=False, total=MAX_B,
                                    desc=f"  sens:{tag} ep{ep+1}"):
                if count >= MAX_B: break
                ctx, future = ctx.to(cfg.device, non_blocking=True), future.to(cfg.device, non_blocking=True)
                for pg in opt.param_groups: pg["lr"] = get_lr(s, total, cfg2)
                opt.zero_grad(set_to_none=True)
                with autocast(AMP_DEVICE, enabled=(cfg2.use_amp and cfg.device == "cuda")):
                    ar_logits, jp, jt = model(ctx, future, use_guidance=True)
                    loss, *_ = compute_loss(ar_logits, jp, jt, ctx, cfg2)
                sc.scale(loss).backward(); sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg2.grad_clip)
                sc.step(opt); sc.update(); model.update_ema()
                count += 1; s += 1
        ppl_g, ppl_b = evaluate_both(model, val_loader, cfg2)
        print(f"  {tag:<35} guided={ppl_g:.2f}  base={ppl_b:.2f}  Δ={ppl_b-ppl_g:+.2f}")
        return ppl_g, ppl_b

    # (a) JEPA dim
    print("\n(a) JEPA embedding dimension:")
    for dim in [64, 128, 256]:
        results[f"jepa_dim_{dim}"] = quick_train({"jepa_dim": dim}, f"jepa_dim={dim}")

    # (b) Pooling strategy
    print("\n(b) Pooling strategy for JEPA summary:")
    for pool in ["mean", "last"]:
        results[f"pool_{pool}"] = quick_train({"jepa_pool": pool}, f"pool={pool}")

    # (c) Guidance injection depth
    print("\n(c) Guidance injection layers:")
    for layers in [[5], [2, 5], [1, 3, 5]]:
        tag = f"layers={layers}"
        results[tag] = quick_train({"guidance_layers": layers}, tag)

    return results

# ─────────────────────────────────────────────────────────────────────────────
# 14.  PARAMETER COUNT TABLE
# ─────────────────────────────────────────────────────────────────────────────
def print_param_table(cfg):
    print("\n" + "─"*60)
    print("PARAMETER COUNT COMPARISON")
    print("─"*60)
    for mode in ["ar_only", "jart", "ar_xattn", "ar_rand"]:
        m = JARTModel(cfg, mode=mode)
        total, trainable = m.count_params()
        # EMA target encoder not trainable — show separately
        ema_params = 0
        if mode == "jart":
            ema_params = sum(p.numel() for p in m._target_params())
        print(f"  {mode:<12}  total={total:>10,}  "
              f"trainable={trainable:>10,}  "
              f"ema(frozen)={ema_params:>9,}")
    print("─"*60)
    print("  Note: ar_xattn and ar_rand have the same trainable params as")
    print("  JART's guidance module. AR-only has fewer (no guidance at all).")

# ─────────────────────────────────────────────────────────────────────────────
# 15.  INFERENCE TIMING
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def benchmark_inference(cfg, n_reps=50):
    """Reports forward-pass latency for each variant."""
    print("\n" + "─"*60)
    print("INFERENCE LATENCY BENCHMARK")
    print("─"*60)
    dummy_ctx = torch.randint(0, cfg.vocab_size,
                               (cfg.batch_size, cfg.seq_len)).to(cfg.device)
    for mode in ["ar_only", "jart", "ar_xattn"]:
        m = JARTModel(cfg, mode=mode).to(cfg.device).eval()
        # warmup
        for _ in range(5):
            m(dummy_ctx, use_guidance=(mode != "ar_only"))
        if cfg.device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_reps):
            m(dummy_ctx, use_guidance=(mode != "ar_only"))
        if cfg.device == "cuda": torch.cuda.synchronize()
        ms = (time.perf_counter() - t0) / n_reps * 1000
        total, trainable = m.count_params()
        print(f"  {mode:<12}  {ms:.2f} ms/batch  "
              f"(trainable params: {trainable:,})")
    print("─"*60)

# ─────────────────────────────────────────────────────────────────────────────
# 16.  PLOTS
# ─────────────────────────────────────────────────────────────────────────────
def plot_ablation_summary(results, outfile="ablation_v6.png"):
    """Bar chart of mean ± std PPL for all variants."""
    labels = ["AR-only", "JART\n(full)", "AR+XAttn\n(param-matched)",
              "AR+Rand\n(noise ctrl)", "JART\nguide off"]
    keys   = ["ar_only", "jart", "ar_xattn", "ar_rand", "jepa_noguide"]
    means  = [results[k][0] for k in keys]
    stds   = [results[k][1] for k in keys]
    colors = ["#E05C5C", "#4A90D9", "#A0C4FF", "#BDB2FF", "#85C1E9"]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels, means, yerr=stds, capsize=5,
                  color=colors, edgecolor="white", linewidth=1.2)
    # annotate mean values
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 1,
                f"{m:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_ylabel("Val PPL (lower = better)", fontsize=11)
    ax.set_title("JART v6 — Full Ablation (mean ± std, 3 seeds)", fontsize=12)
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_ylim(0, max(means) * 1.15)
    plt.tight_layout()
    plt.savefig(outfile, dpi=150)
    print(f"Ablation plot → {outfile}")

def plot_training_curves(history, label, outfile=None):
    if not history["step"]: return
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"JART v6 — {label} Training Curves", fontsize=12)
    s = history["step"]
    axes[0].plot(s, history["train_ppl"],  label="Train", color="#4A90D9", lw=1.5)
    axes[0].plot(s, history["ppl_guided"], label="Val (guided)", color="#27AE60", lw=2)
    axes[0].plot(s, history["ppl_base"],   label="Val (base)",   color="#E05C5C", lw=1.5, ls="--")
    axes[0].set_title("Perplexity ↓"); axes[0].set_xlabel("Step")
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
    axes[1].bar(range(len(history["delta"])), history["delta"],
                color=["#4A90D9" if d > 0 else "#E05C5C" for d in history["delta"]])
    axes[1].axhline(0, color="black", lw=0.8)
    axes[1].set_title("Guidance Δ per eval step\n(base − guided, +ve = helps) ↑")
    axes[1].grid(True, alpha=0.3, axis="y")
    axes[2].plot(s, history["val_align"], color="#8E44AD", lw=1.5, label="Align")
    axes[2].plot(s, history["val_vc"],    color="#E67E22", lw=1.5, label="VICReg")
    axes[2].set_title("JEPA Losses ↓"); axes[2].set_xlabel("Step")
    axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    out = outfile or f"training_v6_{label}.png"
    plt.savefig(out, dpi=150)
    print(f"Training plot → {out}")

# ─────────────────────────────────────────────────────────────────────────────
# 17.  SAVE RESULTS
# ─────────────────────────────────────────────────────────────────────────────
def save_results(results_dict, filename="jart_v6_results.json"):
    out = {}
    for k, v in results_dict.items():
        if isinstance(v, tuple):
            mean, std, ppls = v
            out[k] = {"mean": round(mean, 4), "std": round(std, 4),
                      "per_seed": [round(p, 4) for p in ppls]}
        else:
            out[k] = v
    with open(filename, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Results → {filename}")

# ─────────────────────────────────────────────────────────────────────────────
# 18.  MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    t0 = time.time()

    # ── Data ────────────────────────────────────────────────────────────
    print("\n── Loading WikiText-103 ─────────────────────────────────────")
    train_text, val_text = load_wikitext103(max_train_rows=50000)
    print(f"  Train chars: {len(train_text):,}   Val chars: {len(val_text):,}")

    print("\n── Building tokenizer ───────────────────────────────────────")
    tokenizer = build_tokenizer(train_text, CFG.vocab_size)
    CFG.vocab_size = tokenizer.get_vocab_size()
    print(f"  Vocab: {CFG.vocab_size}")

    print("\n── Tokenising & chunking ─────────────────────────────────────")
    train_chunks = tokenize_and_chunk(train_text, tokenizer, CFG.seq_len)
    val_chunks   = tokenize_and_chunk(val_text,   tokenizer, CFG.seq_len)
    print(f"  Train pairs: {len(train_chunks)-1:,}   Val pairs: {len(val_chunks)-1:,}")

    train_loader = DataLoader(
        PairDataset(train_chunks), batch_size=CFG.batch_size,
        shuffle=True, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=2, drop_last=True)
    val_loader = DataLoader(
        PairDataset(val_chunks), batch_size=CFG.batch_size,
        shuffle=False, num_workers=2, pin_memory=True,
        persistent_workers=True, drop_last=False)

    # ── Parameter counts ─────────────────────────────────────────────────
    print_param_table(CFG)

    # ── Inference latency ────────────────────────────────────────────────
    if CFG.device == "cuda":
        benchmark_inference(CFG)

    # ── Full JART training run (main result) ─────────────────────────────
    print("\n── Main Training Run: JART ──────────────────────────────────")
    jart_model, jart_history, jart_best_ppl = train_model(
        CFG, train_loader, val_loader, tokenizer,
        mode="jart", seed=42, tag="_main")
    plot_training_curves(jart_history, "JART_main", "training_v6_main.png")

    # ── Train/inference gap quantification ───────────────────────────────
    print("\n── Train/Inference Gap Analysis ─────────────────────────────")
    ppl_self, ppl_oracle = evaluate_oracle_vs_self(
        jart_model, val_loader, CFG)
    print(f"  Self-predicted (inference mode): {ppl_self:.2f}")
    print(f"  Oracle future  (upper bound):    {ppl_oracle:.2f}")
    print(f"  Gap:                             {ppl_oracle - ppl_self:+.2f} PPL")
    print(f"  (negative gap = self-pred outperforms oracle, positive = gap to close)")

    # ── Full ablation (5 variants × 3 seeds) ─────────────────────────────
    print("\n── Running Full Ablation ────────────────────────────────────")
    ablation_results = run_full_ablation(
        train_loader, val_loader, CFG, n_seeds=3, epochs=4)
    plot_ablation_summary(ablation_results, "ablation_v6.png")

    # ── Sensitivity ablations ─────────────────────────────────────────────
    print("\n── Sensitivity Ablations ────────────────────────────────────")
    sens_results = run_sensitivity_ablations(train_loader, val_loader, CFG)

    # ── Sample generation ─────────────────────────────────────────────────
    print("\n── Sample Generation ────────────────────────────────────────")
    prompts = ["The history of science", "In the early 20th century",
               "The political system of", "During the Second World"]
    for prompt in prompts[:2]:
        pid = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long)
        with_g = tokenizer.decode(
            jart_model.generate(pid, max_new=60, use_guidance=True))
        without = tokenizer.decode(
            jart_model.generate(pid, max_new=60, use_guidance=False))
        print(f"\n  Prompt: \"{prompt}\"")
        print(f"  [JART]:     {with_g}")
        print(f"  [backbone]: {without}")

    # ── Save all results ──────────────────────────────────────────────────
    all_results = {
        **{k: v for k, v in ablation_results.items()},
        "sensitivity": {k: list(v) for k, v in sens_results.items()},
        "train_inference_gap": {
            "ppl_self_predicted": round(ppl_self, 4) if ppl_self else None,
            "ppl_oracle":         round(ppl_oracle, 4) if ppl_oracle else None,
        },
        "main_run_best_ppl": round(jart_best_ppl, 4),
        "config": {k: str(v) for k, v in CFG.__dict__.items()
                   if not callable(v) and k != "seeds"},
    }
    save_results(all_results, "jart_v6_results.json")

    print(f"\n── Done in {(time.time()-t0)/60:.1f} min ─────────────────────────────")
    print("\nOutput files:")
    print("  best_jart_main_s42.pt   — main JART model weights")
    print("  training_v6_main.png    — training curves")
    print("  ablation_v6.png         — 5-variant ablation (mean ± std)")
    print("  jart_v6_results.json    — all numerical results")

Device: cuda

── Loading WikiText-103 ─────────────────────────────────────


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

  pool=mean                           guided=2254.47  base=3178.74  Δ=+924.28


  pool=last                           guided=2270.26  base=3178.37  Δ=+908.11

(c) Guidance injection layers:


  layers=[5]                          guided=2254.47  base=3178.74  Δ=+924.28


  layers=[2, 5]                       guided=3303.33  base=3304.71  Δ=+1.38


  layers=[1, 3, 5]                    guided=3202.34  base=3202.57  Δ=+0.24

── Sample Generation ────────────────────────────────────────

  Prompt: "The history of science"
  [JART]:     The history of science fiction , Dylan also published in 2011 . According to Carey ' s novel The Guardian , " In Chains ' first film written by critics and writer producer Michael Jackson — writing : " For the lyrics are a simple music , I wrote that she had been inspired with her book , which has praised it as well
  [backbone]: The history of science @-@ fiction was built by the American Government at London ' s Day , which is a single single in North America . In August 1919 , she continued to form several months later after their release and began with this film from the same year before being sent into an early 1940s in 1996 . It was found on

  Prompt: "In the early 20th century"
  [JART]:     In the early 20th century , Missoula has recently developed the church ' s first population . The site